# 06 — Streamlit demo on Kaggle (with public Cloudflare tunnel)

Runs `app/streamlit_app.py` inside a Kaggle session and exposes it via a free Cloudflare tunnel —
no ngrok account, no auth token, no port-forwarding setup.

**Setup before running**:
1. **Settings → Accelerator → GPU P100** (or T4 ×2) — needed to load Qwen2.5-7B in 4-bit.
2. **Settings → Internet → On**.
3. **Add-ons → Secrets → `HF_TOKEN`** attached (Read scope is enough — we only pull the LoRA adapter).
4. **Run all cells.**

After the last cell, you'll get a public URL like `https://<random>.trycloudflare.com` —
share it with anyone (judges, classmates) for the live demo. The URL stays alive for as long as the
Kaggle session runs (max 12 h).

In [ ]:
# 1. Clone (or update) the repo at /kaggle/working/LawMate.
import os, subprocess

REPO_URL = "https://github.com/tamir39/rag-llm-vietnam-law-advisor.git"
REPO_DIR = "/kaggle/working/LawMate"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "develop", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "origin", "develop"])

os.chdir(REPO_DIR)
print(subprocess.check_output(["git", "log", "-1", "--oneline"]).decode().strip())

In [ ]:
# 2. Install the LLM/RAG stack + Streamlit. Kaggle has torch/transformers/datasets baseline.
%pip install -q -U "peft>=0.12" "trl>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" "sentence-transformers>=3.0" "faiss-cpu>=1.8" "streamlit>=1.36"

In [ ]:
# 3. HF login — needed to pull Tamir39/qwen2_5-7b-vietnam-tax-lora (and Qwen base if private/gated).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF login OK")

In [ ]:
# 4. Build the FAISS index if it isn't there yet (~30s on Kaggle CPU).
import os, subprocess

if not os.path.isfile("experiments/index/kb.faiss"):
    subprocess.check_call(["python", "scripts/build_index.py"])
else:
    print("FAISS index already present — skipping build")

In [ ]:
# 5. Download the cloudflared binary (one-time, ~30 MB).
import os, stat, subprocess

BIN = "/kaggle/working/cloudflared"
if not os.path.isfile(BIN):
    subprocess.check_call([
        "wget", "-q", "-O", BIN,
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ])
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
print(subprocess.check_output([BIN, "--version"]).decode().strip())

In [ ]:
# 6. Launch Streamlit + cloudflared tunnel. Public URL prints at the end.
import os, re, subprocess, time

os.chdir("/kaggle/working/LawMate")

# Start Streamlit in background
streamlit = subprocess.Popen(
    [
        "streamlit", "run", "app/streamlit_app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.address", "0.0.0.0",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

print("[streamlit] starting...")
started = False
for _ in range(120):
    line = streamlit.stdout.readline()
    if not line:
        time.sleep(0.25)
        continue
    print(line.rstrip())
    if "You can now view your Streamlit app" in line or "Network URL" in line:
        started = True
        break
if not started:
    raise RuntimeError("Streamlit failed to come up — see log above")

# Open cloudflared tunnel
print("\n[cloudflared] opening tunnel...")
tunnel = subprocess.Popen(
    ["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
for _ in range(180):
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    print(line.rstrip())
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError("Could not parse cloudflared URL from output")

print("\n" + "=" * 72)
print(f"  DEMO URL:  {public_url}")
print("=" * 72)
print("\nFirst question loads Qwen2.5-7B + LoRA adapter (~2-3 min).")
print("After that, each answer takes ~10-30 s on P100.")
print("Stop the demo by interrupting this cell or stopping the kernel.")

## Stopping the demo

- **Interrupt the cell above** (◼ button) — this kills both Streamlit and the tunnel.
- Or **Stop Kernel** from the right sidebar to shut down the whole session.
- Closing the Kaggle browser tab does **not** stop the session — your tunnel keeps running
  (and counts against your 30 GPU-hr/week quota) until the kernel idles out (~20 min).

## Troubleshooting

- **Tunnel URL prints but page shows error** — wait ~10 s and reload; cloudflared takes a moment
  to register the route after printing the URL.
- **"Address already in use" on port 8501** — a previous Streamlit didn't shut down. Restart the
  kernel (right sidebar → Stop session → Start) and re-run from cell 1.
- **OOM when loading Qwen** — switch accelerator to T4 ×2 (30 GB combined VRAM).